In [ ]:
import os
import ROOT
import array

In [ ]:
def generate_neutrino_energy(n_events=10000, out_file="neutrino_energy.root"):

    MEAN  = 3.5   # GeV
    SIGMA = 1.0   # GeV

    # ── Open output file ──────────────────────────────────────────────────────
    f = ROOT.TFile.Open(out_file, "RECREATE")
    if not f or f.IsZombie():
        print(f"[ERROR] Cannot open output file: {out_file}")
        sys.exit(1)

    # ── TTree construction ────────────────────────────────────────────────────
    tree = ROOT.TTree("NeutrinoTree", "Simulated Neutrino Energy Data")

    # In PyROOT a TTree branch needs a mutable buffer to write into.
    # array.array('d', [0.0]) creates a one-element C-style double array —
    # this is the standard PyROOT idiom that mirrors Double_t NeutrinoEnergy
    # and the & (address-of) operator in the C++ version.
    NeutrinoEnergy = array.array('d', [0.0])   # 'd' = C double = Double_t
    tree.Branch("NeutrinoEnergy", NeutrinoEnergy, "NeutrinoEnergy/D")

    # ── Random number generator ───────────────────────────────────────────────
    # TRandom3: Mersenne Twister — the HEP standard
    rng = ROOT.TRandom3(42)   # seed=42 for reproducibility; use 0 for random

    # ── Fill loop ─────────────────────────────────────────────────────────────
    for i in range(n_events):
        NeutrinoEnergy[0] = rng.Gaus(MEAN, SIGMA)   # write into buffer index [0]
        tree.Fill()

    # ── Write & close ─────────────────────────────────────────────────────────
    f.Write()
    f.Close()

    print(f"[INFO] Done. Wrote {n_events:,} events to '{out_file}'")
    print(f"       TTree: NeutrinoTree  |  Branch: NeutrinoEnergy  |  mean={MEAN}, sigma={SIGMA}")


In [ ]:
generate_neutrino_energy(2000)

In [ ]:
def get_histo(in_file="neutrino_energy.root"):
    if not os.path.exists(in_file):
        print(f"[ERROR] File not found: {in_file}")
        return None

    f = ROOT.TFile.Open(in_file, "READ")
    if not f or f.IsZombie():
        print(f"[ERROR] Cannot open file: {in_file}")
        return None

    tree = f.Get("NeutrinoTree")
    if not tree:
        print(f"[ERROR] TTree 'NeutrinoTree' not found in {in_file}")
        f.Close()
        return None

    n_entries = tree.GetEntries()
    print(f"[INFO] Opened '{in_file}' — {n_entries:,} events found.")

    NeutrinoEnergy = array.array('d', [0.0])
    tree.SetBranchAddress("NeutrinoEnergy", NeutrinoEnergy)

    h = ROOT.TH1D("h_NeutrinoEnergy",
                  "Neutrino Energy Distribution;"
                  "Neutrino Energy [GeV];"
                  "Events / 0.09 GeV",
                  100, 0.0, 10.0)
    h.SetDirectory(0)                        # ← the critical line
    h.SetLineColor(ROOT.kBlue + 1)
    h.SetLineWidth(2)
    h.SetFillColorAlpha(ROOT.kBlue - 9, 0.45)

    for i in range(n_entries):
        tree.GetEntry(i)
        h.Fill(NeutrinoEnergy[0])

    f.Close()
    return h

In [ ]:
# ── Draw in notebook ──────────────────────────────────────────
h = get_histo()

c = ROOT.TCanvas("c_NeutrinoEnergy", "Neutrino Energy", 900, 650)
c.SetGrid()
c.SetLeftMargin(0.12)
c.SetBottomMargin(0.12)
h.Draw("HIST")
c.Draw()